# Assignment 2 — ASR Decoding Evaluation

**Быстрая проверка:** `MAX_SAMPLES = 10`

In [1]:
# ── 1. Установка зависимостей ──────────────────────────────────────────────
!pip install -q https://github.com/kpu/kenlm/archive/master.zip
!pip install -q jiwer transformers torch torchaudio soundfile pandas matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.6/553.6 kB 11.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 117.5 MB/s eta 0:00:00


In [2]:
# ── 2. Клонирование репозитория ────────────────────────────────────────────
# Замени URL на адрес своего публичного репозитория на GitHub
REPO_URL = "https://github.com/karinaDen/ai-talent-hub-itmo-speech-course.git"

!git clone {REPO_URL} repo
%cd repo/assignments/assignment2

Cloning into 'repo'...
remote: Enumerating objects: 653, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (87/87), done.
remote: Total 653 (delta 47), reused 78 (delta 36), pack-reused 530 (from 1)
Receiving objects: 100% (653/653), 130.43 MiB | 14.69 MiB/s, done.
Resolving deltas: 100% (77/77), done.
Updating files: 100% (453/453), done.
/content/repo/assignments/assignment2


In [3]:
# ── 3. Распаковка LM (если .arpa.gz) ──────────────────────────────────────
import os, gzip, shutil

gz_path = 'lm/3-gram.pruned.1e-7.arpa.gz'
arpa_path = 'lm/3-gram.pruned.1e-7.arpa'

if os.path.exists(gz_path) and not os.path.exists(arpa_path):
    print('Decompressing LM...')
    with gzip.open(gz_path, 'rb') as f_in, open(arpa_path, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
    print('Done.')
elif os.path.exists(arpa_path):
    print('LM already decompressed.')
else:
    print('ERROR: LM file not found!')

LM already decompressed.


In [4]:
# ── 4. Конфигурация ────────────────────────────────────────────────────────
MAX_SAMPLES = 200   # поставь 10 для быстрой проверки
BEAM_WIDTH  = 10
LM_PATH     = 'lm/3-gram.pruned.1e-7.arpa'

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}  |  MAX_SAMPLES: {MAX_SAMPLES}')

Device: cuda  |  MAX_SAMPLES: 200


In [5]:
# ── 5. Быстрая санитарная проверка на примерах ─────────────────────────────
from wav2vec2decoder import Wav2Vec2Decoder, test

dec = Wav2Vec2Decoder(lm_model_path=LM_PATH, beam_width=BEAM_WIDTH, alpha=0.1, beta=0.5)

test(dec, 'examples/sample1.wav',
     'if you are generous here is a fitting opportunity for the exercise of your magnanimity '
     'if you are proud here am i your rival ready to acknowledge myself your debtor for an act '
     'of the most noble forbearance')
test(dec, 'examples/sample6.wav', 'at this time all participants are in a listen only mode')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/358 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


REF : if you are generous here is a fitting opportunity for the exercise of your magnanimity if you are proud here am i your rival ready to acknowledge myself your debtor for an act of the most noble forbearance


model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

  [greedy] if you are generous here is a fiting opportunity for the exercise of your menanimiti if you are proud here am i ar rival rather toknowledge myself your diptor for a act of the most noble forbarans
           WER=23.68%  CER=10.24%
  [beam] if you are generous here is a fiting opportunity for the exercise of your menanimity if you are proud here am i ar rival rather toknowledge myself your diptor for an act of the most noble forbarans
           WER=21.05%  CER=9.27%
  [beam_lm] if you are generous here is a fiting opportunity for the exercise of your menanimiti if you are proud here am i ar rival rather toknowledge myself your diptor for a act of the most noble forbarans
           WER=23.68%  CER=10.24%
  [beam_lm_rescore] if you are generous here is a fiting opportunity for the exercise of your menanimity if you are proud here am i ar rival rather toknowledge myself your diptor for an act of the most noble forbarans
           WER=21.05%  CER=9.27%
REF : at this time all p

In [ ]:
# ── 6. Полная оценка (Tasks 1–7b) ──────────────────────────────────────────
!python evaluate.py \
    --data-root . \
    --lm {LM_PATH} \
    --beam-width {BEAM_WIDTH} \
    --max-samples {MAX_SAMPLES}

In [ ]:
# ── 7. Показать сохранённые графики ───────────────────────────────────────
from IPython.display import Image, display
import glob

for path in sorted(glob.glob('results/*.png')):
    print(f'\n--- {os.path.basename(path)} ---')
    display(Image(path))

In [6]:
# ── 8. (Task 8) Обучение финансового KenLM ─────────────────────────────────
# Только в Colab (Linux): строим KenLM CLI и обучаем 3-gram на earnings22_train/corpus.txt

!apt-get install -q cmake libboost-all-dev
!git clone --depth=1 https://github.com/kpu/kenlm /tmp/kenlm_build
!mkdir -p /tmp/kenlm_build/build
!cmake -S /tmp/kenlm_build -B /tmp/kenlm_build/build -DCMAKE_BUILD_TYPE=Release
!make -C /tmp/kenlm_build/build -j2 lmplz build_binary

!mkdir -p lm
!/tmp/kenlm_build/build/bin/lmplz -o 3 --discount_fallback \
    < data/earnings22_train/corpus.txt \
    > /tmp/financial-3gram.arpa

# Используем как plain .arpa (kenlm Python не требует gzip в Colab)
!cp /tmp/financial-3gram.arpa lm/financial-3gram.arpa
print('Financial LM ready: lm/financial-3gram.arpa')

Reading package lists...
Building dependency tree...
Reading state information...
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
The following additional packages will be installed:
  javascript-common libboost-atomic-dev libboost-atomic1.74-dev
  libboost-atomic1.74.0 libboost-chrono-dev libboost-chrono1.74-dev
  libboost-chrono1.74.0 libboost-container-dev libboost-container1.74-dev
  libboost-container1.74.0 libboost-context-dev libboost-context1.74-dev
  libboost-context1.74.0 libboost-coroutine-dev libboost-coroutine1.74-dev
  libboost-coroutine1.74.0 libboost-date-time-dev libboost-date-time1.74-dev
  libboost-date-time1.74.0 libboost-exception-dev libboost-exception1.74-dev
  libboost-fiber-dev libboost-fiber1.74-dev libboost-fiber1.74.0
  libboost-filesystem-dev libboost-filesystem1.74-dev
  libboost-filesystem1.74.0 libboost-graph-dev libboost-graph-parallel-dev
  libboost-graph-parallel1.74-dev libboost-graph-parallel1.74.0
  libboost-graph1.74-dev libboost-gr

In [7]:
# ── 9. (Task 9) Сравнение LibriSpeech LM vs Financial LM ──────────────────
import csv, jiwer
from wav2vec2decoder import Wav2Vec2Decoder
import torchaudio

def load_manifest(path, max_n=MAX_SAMPLES):
    rows = []
    with open(path) as f:
        for r in csv.DictReader(f):
            rows.append((r['path'], r['text']))
            if len(rows) >= max_n:
                break
    return rows

def eval_set(decoder, samples, method):
    hyps, refs = [], []
    for path, ref in samples:
        audio, _ = torchaudio.load(path)
        hyps.append(decoder.decode(audio, method))
        refs.append(ref)
    return jiwer.wer(refs, hyps), jiwer.cer(refs, hyps)

ls  = load_manifest('data/librispeech_test_other/manifest.csv')
e22 = load_manifest('data/earnings22_test/manifest.csv')

results = []
for lm_name, lm_path in [('LibriSpeech 3-gram', 'lm/3-gram.pruned.1e-7.arpa'),
                           ('Financial 3-gram',  'lm/financial-3gram.arpa')]:
    for method in ['beam_lm', 'beam_lm_rescore']:
        dec = Wav2Vec2Decoder(lm_model_path=lm_path, beam_width=BEAM_WIDTH, alpha=0.1, beta=0.5)
        wer_ls, cer_ls   = eval_set(dec, ls,  method)
        wer_e22, cer_e22 = eval_set(dec, e22, method)
        row = {'LM': lm_name, 'Method': method,
               'LS WER': f'{wer_ls:.4f}', 'LS CER': f'{cer_ls:.4f}',
               'E22 WER': f'{wer_e22:.4f}', 'E22 CER': f'{cer_e22:.4f}'}
        results.append(row)
        print(row)

import pandas as pd
pd.DataFrame(results).to_csv('results/task9_lm_comparison.csv', index=False)
print('Saved: results/task9_lm_comparison.csv')

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'LM': 'LibriSpeech 3-gram', 'Method': 'beam_lm', 'LS WER': '0.1122', 'LS CER': '0.0381', 'E22 WER': '0.5539', 'E22 CER': '0.2552'}


Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'LM': 'Financial 3-gram', 'Method': 'beam_lm', 'LS WER': '0.1127', 'LS CER': '0.0385', 'E22 WER': '0.5274', 'E22 CER': '0.2532'}


Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'LM': 'Financial 3-gram', 'Method': 'beam_lm_rescore', 'LS WER': '0.1107', 'LS CER': '0.0378', 'E22 WER': '0.5389', 'E22 CER': '0.2522'}
Saved: results/task9_lm_comparison.csv
